<a href="https://colab.research.google.com/github/PadmaPrabhasKorsipati/CSA63-Threat-Intelligence-and-Network-Security/blob/main/Lab/Unit%204%20Experiments/LAB_08_Zero%20Trust%20Continuous%20Verification%20Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
def zero_trust_authorize(request, resource_policy):
    reasons = []

    allowed_users = resource_policy.get(request["resource"], set())

    if request["user"] not in allowed_users:
        reasons.append("user not authorized for this resource")

    if not request["mfa_passed"]:
        reasons.append("MFA not completed")

    if not request["device_posture"]["antivirus_enabled"]:
        reasons.append("antivirus disabled")

    if not request["device_posture"]["os_patched"]:
        reasons.append("OS not fully patched")

    return {
        "granted": len(reasons) == 0,
        "reasons": reasons
    }


def test_experiment8():
    policy = {
        "finance_db": {"csmith", "afinance"}
    }

    healthy_request = {
        "user": "csmith",
        "mfa_passed": True,
        "device_posture": {
            "antivirus_enabled": True,
            "os_patched": True
        },
        "resource": "finance_db"
    }

    assert zero_trust_authorize(
        healthy_request,
        policy
    )["granted"] is True

    unhealthy_request = dict(
        healthy_request,
        device_posture={
            "antivirus_enabled": False,
            "os_patched": True
        }
    )

    result = zero_trust_authorize(unhealthy_request, policy)

    assert result["granted"] is False
    assert "antivirus disabled" in result["reasons"]

    unauthorized_request = dict(
        healthy_request,
        user="attacker99"
    )

    assert zero_trust_authorize(
        unauthorized_request,
        policy
    )["granted"] is False

    print("All test cases passed.")


test_experiment8()



All test cases passed.
